# Day 2 Starter Kit: Complete the Grounding

**Your dataset: Risk, incident & problem (code RISK).** A technology organisation's service-management and risk log: incidents, problem investigations, the risk register, changes, vendors, alerts and audits.

**The idea.** Retrieval finds records that *sound* similar. It cannot tell whether two records describe the *same event*. The graph can, **if it knows which parts of a record identify an event.** That knowledge is a list of regular expressions, and writing it well is your job.

**What is given and working:** retrieval, the baseline RAG, graph building from a *pattern registry*, and the grounded answer step. The registry has **one pattern** (Incident ID).

**What you do**
0. **Plug in your own LLM.** Add your key from any provider and, if needed, edit the one `call_llm` function so the notebook runs on it. Nothing else in the notebook touches the provider.
1. Explore the data and decide which other patterns identify a single event (**Kind 1**) and which are incidental fields reused across unrelated events (**Kind 2**: a room number, an amount, a semester...).
2. Add your patterns to the registry, each with a reason. List the lookalikes you *rejected*, and why.
3. Run the public question pack against baseline and grounded. Explain every outcome.
4. Say where the baseline fails and your graph works, and **one failure you still have.**

**No Neo4j?** If your team cannot use Neo4j Aura, replace the graph in sections 7 and 8 with lookup tables of your own (the Student Setup Guide shows the structure). Everything else, including the judging, is the same.

**What you submit:** this notebook, run top to bottom on a fresh runtime. The judges read your registry, your outcomes table and your remaining-failure note, then run a hidden question pack against your graph.

> Linking on the wrong thing is as bad as linking on nothing: a pattern that links unrelated records makes the system *invent* connections. Add a pattern only if you can say why two records sharing it are the same event.

## 1. Setup

In [ ]:
%pip install -q sentence-transformers hnswlib openai python-dotenv neo4j

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import hnswlib
import os

def get_secret(name):
    """Reads a secret from Colab's Secrets manager (padlock icon in the left
    sidebar) if running on Colab; falls back to a local .env file otherwise."""
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.getenv(name)

model = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model ready")

### Your LLM provider

Bring your own key (any provider) and add it to Colab's Secrets manager (padlock icon) as `LLM_API_KEY`. Every LLM call in this notebook goes through the single function `call_llm` below.

Run the cell: it should print a one-word reply. If it errors, fix this cell before going on.

In [ ]:
# ---- YOUR LLM PROVIDER (your own key, any provider) ----
# Add your key to Colab Secrets as LLM_API_KEY. The whole notebook talks to the model through
# ONE function, call_llm(prompt) -> str, so this is the only cell you change for a new provider.
# The default below works for OpenAI and for any provider with an OpenAI-compatible endpoint:
# set LLM_MODEL (and LLM_BASE_URL if it is not OpenAI) from your provider's documentation.
# A provider with its own SDK: keep the name and signature of call_llm, rewrite the body.
from openai import OpenAI

LLM_MODEL = "gpt-4o-mini"
LLM_BASE_URL = None  # None = OpenAI. Otherwise your provider's OpenAI-compatible URL

client = OpenAI(api_key=get_secret("LLM_API_KEY"), base_url=LLM_BASE_URL)

def call_llm(prompt):
    """Sends one prompt, returns the model's reply as plain text.
    Keep temperature=0: otherwise the same question can give a different answer on each run."""
    response = client.chat.completions.create(
        model=LLM_MODEL, temperature=0,
        messages=[{"role": "user", "content": prompt}])
    return response.choices[0].message.content

print("LLM check:", call_llm("Reply with the single word: ready"))

### Connect to Neo4j Aura

Create a free [Neo4j Aura](https://neo4j.com/cloud/aura-free/) instance if you haven't, then add `NEO4J_URI`, `NEO4J_USER` and `NEO4J_PASSWORD` to Colab's Secrets manager (padlock icon). If you hit `ServiceUnavailable`, the instance auto-paused: re-run this cell.

**Cannot use Neo4j?** Skip this cell. Replace the graph-building cell (section 7) and `graph_check` (section 8) with lookup-table versions that keep the same return shape. The Student Setup Guide shows the table structure.

In [ ]:
from neo4j import GraphDatabase

uri = get_secret("NEO4J_URI")
user = get_secret("NEO4J_USER")
password = get_secret("NEO4J_PASSWORD")

driver = GraphDatabase.driver(uri, auth=(user, password))

def run_write(query, **params):
    with driver.session() as session:
        session.run(query, **params)

def run_read(query, **params):
    with driver.session() as session:
        result = session.run(query, **params)
        return [record.data() for record in result]

print(run_read("RETURN 'Connection successful' AS message")[0]["message"])

## 2. The dataset: Risk, incident & problem

141 records in 12 categories. A technology organisation's service-management and risk log: incidents, problem investigations, the risk register, changes, vendors, alerts and audits. Four pairs of categories deliberately share vocabulary. Most records stand alone; some carry an identifier or a code.

In [ ]:
DATASET_ID = "RISK"
DATASET_NAME = "Risk, incident & problem"

rec_categories = {
    'Incident_Triage': [
        'A user reported that the payment portal was loading slowly',
        'The service desk logged a login failure affecting several staff',
        'An alert on the email gateway was escalated to the on-call engineer',
        'A priority was assigned after the impact assessment was completed',
        'A duplicate ticket for the same outage was merged by the service desk',
        'The major incident bridge was opened for a network disruption',
        'A call about a frozen laptop was categorised as a hardware issue',
        'Customer-facing teams were notified about a degraded service',
        'A ticket was reassigned after being routed to the wrong team',
        'Incident INC4025 was logged when the payment portal stopped accepting card transactions',
        'Incident INC4012 took down the customer chat service for two hours',
        'Severity 2 was assigned to the slow report generation ticket',
        'An outage affected 300 users of the HR portal',
        'Incident INC4057 was caused by overheating on Rack 22',
    ],
    'Incident_Resolution': [
        'The failed batch job was restarted and completed successfully',
        'A faulty switch was replaced and connectivity was restored',
        'The service desk confirmed resolution with the affected user',
        'A workaround was shared with users while a fix was prepared',
        'A rollback restored the previous version of the application',
        'Disk space was cleared on the file server to restore service',
        'The incident was closed after twenty-four hours without recurrence',
        'A certificate was renewed to end the browser warnings',
        'Services were failed over to the standby site',
        'Incident INC4025 was resolved after the gateway configuration was rolled back',
        'Incident INC4033 was resolved with a workaround while the fix was prepared',
        'Incident INC4033 was traced to a loose power cable on Rack 22',
        'Change CHG-5521 was rolled back after the upgrade slowed the reporting queries',
    ],
    'Problem_Analysis': [
        'Recurring timeouts were grouped for a deeper investigation',
        'A known error record was created for the printing issue',
        'Analysts compared logs across three failed deployments',
        'A trend report showed repeated memory spikes on the application servers',
        'The problem manager requested more data from the network team',
        'A workaround was documented against the known error',
        'Related tickets were linked to a single investigation',
        'A problem record was reopened after a repeat failure',
        'Analysts proposed a permanent fix for the intermittent login failure',
        'Problem PRB114 was opened for the recurring timeouts on the booking service',
        'Problem PRB101 was linked to six incident tickets about slow report generation',
        'Problem PRB120 estimated that 450 users were affected by the recurring errors',
    ],
    'Root_Cause_Reviews': [
        "A review meeting examined the cause of last month's outage",
        'The review found that monitoring thresholds had been set too high',
        'Lessons learned were shared with the operations team',
        'A blameless review identified a gap in the handover process',
        'The review recommended adding a second approver for risky deployments',
        'Action items from the review were assigned owners and dates',
        'A review of the storage failure was postponed to next week',
        'The root cause was traced to an expired service account',
        'Review notes were published on the internal knowledge base',
        'Problem PRB114 was traced to an expired service account used by the booking service',
        'Problem PRB120 was reviewed and the action items were assigned owners',
        'Incident INC4012 was detected late because a monitoring alert was misconfigured',
    ],
    'Risk_Register': [
        'A new risk was logged for dependence on a single cloud provider',
        'The risk committee reviewed the top ten risks for the quarter',
        'A risk owner was named for the data retention exposure',
        'Likelihood scores were updated after the latest audit',
        'A risk was re-rated from medium to high',
        'The register was cleaned up by closing duplicate entries',
        'Residual risk was calculated after controls were tested',
        'A new regulatory requirement was added as a risk item',
        'Risk appetite statements were reviewed by the board',
        'Risk R-07 was re-rated from medium to high by the risk committee',
        'Risk R-19 was added for the new data retention rule',
    ],
    'Risk_Mitigation': [
        'Mitigation plan for the key-person dependency was approved',
        'A control was added to reduce the chance of data leakage',
        'Insurance cover was reviewed for cyber-related losses',
        'Backup restoration tests were scheduled every quarter',
        'A risk owner reported progress on the mitigation actions',
        'Funding was approved for a secondary network link',
        'Staff completed training as a mitigation for phishing exposure',
        'A mitigation deadline was extended due to vendor delays',
        'The mitigation was verified by the internal audit team',
        'Risk R-07 mitigation plan was approved with a named owner',
    ],
    'Change_Management': [
        'A change request was approved for the database upgrade',
        'The change advisory board met to review the weekly changes',
        'An emergency change was raised to patch a vulnerable server',
        'A change freeze was announced before the festival sale',
        'A failed change was rolled back within the maintenance window',
        'Change records were updated with post-implementation notes',
        'A standard change template was approved for routine patching',
        'A change was rejected for lacking a back-out plan',
        'Peer review was required for all network configuration changes',
        'Change CHG-5521 was approved to upgrade the database cluster',
        'Change CHG-4098 was an emergency patch for a vulnerable server',
        'Problem PRB101 was closed after the permanent fix went live through a change',
        'The change freeze was extended through the month-end close',
    ],
    'Vendor_Risk': [
        'A vendor security questionnaire was sent for annual review',
        "The payment vendor's financial health was flagged for monitoring",
        'A contract clause on data breaches was renegotiated',
        'A critical supplier was moved to the watch list',
        'A vendor audit report was received and filed',
        'Exit plans were requested from all tier-one vendors',
        'A subcontractor of a vendor was found to be unvetted',
        "The vendor's recovery time commitment was documented",
        'Vendor performance scores were shared with procurement',
        'Risk R-12 covers the unvetted subcontractor of the payment vendor',
        'Risk R-07 was linked to the exit plan requested from the payment vendor',
        'A vendor audit finding was rated Severity 2 by the review team',
    ],
    'Security_Alerts': [
        'An alert flagged repeated failed logins from an unfamiliar location',
        'The security team blocked a suspicious IP address',
        'A phishing email was reported by several employees',
        'Antivirus detected malware on a contractor laptop',
        'A vulnerability scan listed outdated software on two servers',
        'Unusual data transfer volumes triggered an alert',
        'Multi-factor authentication was enforced for admin accounts',
        'A false positive alert was suppressed after review',
        'An access review removed stale accounts',
        'Incident INC4041 was raised after repeated failed logins from an unfamiliar location',
        'A phishing campaign targeted 300 users in the sales team',
        'A cooling alarm was raised for Rack 14 in the server room',
    ],
    'Monitoring': [
        'Dashboards were updated to show database latency',
        'A threshold alert fired for high CPU on a web server',
        'Log retention was extended for the payment systems',
        'Synthetic checks detected a slow checkout page',
        'A monitoring agent was upgraded across all servers',
        'Alert noise was reduced by tuning duplicate rules',
        'Capacity trends were reviewed for the storage arrays',
        'An uptime report was generated for the quarterly review',
        'A blind spot was found in monitoring of the batch scheduler',
        'Rack 14 was powered down for a scheduled firmware update',
        'Batch job failures were most frequent during the month-end close',
    ],
    'Business_Continuity': [
        'The disaster recovery plan was tested with a simulated outage',
        'Contact lists for the crisis team were refreshed',
        'A tabletop exercise covered a prolonged power failure',
        'Recovery time objectives were reviewed for critical systems',
        'Offsite backups were verified for readability',
        'A continuity plan was drafted for the new office',
        'Staff evacuation procedures were rehearsed',
        'The alternate site was inspected for readiness',
        'A dependency map was updated for critical business services',
        'Incident INC4057 triggered the failover to the standby site',
    ],
    'Audit_Compliance': [
        'An internal audit sampled access approvals for the finance system',
        'Evidence was collected for the annual security certification',
        'An auditor requested logs of privileged access',
        'A control deficiency was recorded and assigned for remediation',
        'The compliance calendar was updated for the new year',
        'A policy exception was approved with an expiry date',
        'An external assessment reviewed encryption standards',
        'Segregation of duties conflicts were reported to management',
        'An audit finding was closed after evidence was accepted',
        'Incident INC4041 was included in the evidence for the access review',
        'A licence audit found 450 users with unused accounts',
    ],
}

rec_texts = []
rec_labels = []
for category, sentences in rec_categories.items():
    for s in sentences:
        rec_texts.append(s)
        rec_labels.append(category)

print(f"Dataset {DATASET_ID}: {len(rec_texts)} records, {len(rec_categories)} categories")

## 3. Retrieval (given: this part works)

In [ ]:
rec_embeddings = model.encode(rec_texts, show_progress_bar=True)
rec_embeddings = np.ascontiguousarray(rec_embeddings.astype('float32'))

dim = rec_embeddings.shape[1]
rec_index = hnswlib.Index(space='cosine', dim=dim)
rec_index.init_index(max_elements=len(rec_texts), ef_construction=200, M=16)
rec_index.add_items(rec_embeddings, np.arange(len(rec_texts)))
rec_index.set_ef(50)
print(f"Index built: {len(rec_texts)} records")


def retrieve(question, k=3):
    q_emb = model.encode([question])[0].astype('float32').reshape(1, -1)
    labels, distances = rec_index.knn_query(q_emb, k=k)
    return [(int(idx), rec_texts[idx], rec_labels[idx]) for idx in labels[0]]

## 4. Baseline RAG (given: no grounding)

The plain Week 2 pipeline. It retrieves similar records and trusts the LLM to reason about whether they are connected.

In [ ]:
# The plain Week 2 prompt -- no grounding. temperature=0 so runs are repeatable.
BASELINE_PROMPT = """Answer the question using the context below.

Context:
{context}

Question: {question}

Answer:"""

def baseline_rag_answer(question, k=3):
    results = retrieve(question, k=k)
    context = "\n".join(f"- {text}" for _, text, _ in results)
    prompt = BASELINE_PROMPT.format(context=context, question=question)
    return {"question": question, "retrieved": results, "answer": call_llm(prompt)}

## 5. Explore: what could identify one event?

Read the records below. For each string with digits, ask: *if I saw this same string in a different record, would it still point at the same event?*

In [ ]:
import re

def show_id_like_records(max_rows=80):
    """Prints every record containing a digit. Read them: which of these strings
    would still point at ONE specific event if you saw it in a different record?"""
    n = 0
    for i, t in enumerate(rec_texts):
        if re.search(r"\d", t):
            print(f"R{i:<3} {t}")
            n += 1
            if n >= max_rows:
                break
    print(f"\n{n} records shown")

show_id_like_records()

## 6. YOUR TASK: the pattern registry

Each entry becomes a graph node label. Two records that mention the *same identifier* get connected through that node. Edit this cell, then run it.

In [ ]:
import re

PATTERNS = [
    {
        "name": 'Incident',
        "regex": r"\bIncident\s+(INC\d{4})\b",   # group 1 = the identifier
        "kind": 1,
        "example": 'Incident INC4025',
        "why": 'An incident number names one incident from logging to resolution.',
    },
    # YOUR PATTERNS GO HERE. One entry per pattern, same five fields as above.
    # Not sure a pattern is right? Test it first:
    #   re.findall(r"your regex", campus_texts[i])   or loop over all campus_texts.
    # {
    #     "name": "Xyz",                      # becomes a node label in Neo4j
    #     "regex": r"...",                        # ONE capture group around the identifier
    #     "kind": 1,                              # 1 = names ONE event/person; 2 = incidental field
    #     "example": "...",                       # a string from the data it matches
    #     "why": "...",                           # why two records sharing it ARE the same event
    # },
]

REJECTED_CANDIDATES = [
    # Things that LOOK like identifiers but you decided NOT to link on. Credit is given for good reasons.
    # {"candidate": "...", "kind": 2, "why_rejected": "..."},
]

def validate_registry():
    """Checks that every pattern is complete and compiles. Run this before building the graph."""
    problems = []
    for p in PATTERNS:
        label = p.get("name", "?")
        for key in ("name", "regex", "kind", "example", "why"):
            if p.get(key) in (None, ""):
                problems.append(f"{label}: missing '{key}'")
        if p.get("name") and not re.fullmatch(r"[A-Za-z][A-Za-z0-9_]*", p["name"]):
            problems.append(f"{label}: name must be letters/digits/underscore only")
        try:
            rx = re.compile(p.get("regex", ""))
            if rx.groups < 1:
                problems.append(f"{label}: regex needs ONE capture group around the identifier")
        except re.error as e:
            problems.append(f"{label}: regex does not compile ({e})")
        if p.get("kind") not in (1, 2):
            problems.append(f"{label}: kind must be 1 or 2")
    return problems

def show_register():
    print(f"{'Pattern':<10} {'Kind':<5} {'Example':<20} Why")
    for p in PATTERNS:
        print(f"{p['name']:<10} {p['kind']:<5} {p['example']:<20} {p['why']}")
    if REJECTED_CANDIDATES:
        print("\nConsidered but rejected:")
        for r in REJECTED_CANDIDATES:
            print(f"  - {r['candidate']} (Kind {r['kind']}): {r['why_rejected']}")

problems = validate_registry()
print("Registry problems:", problems if problems else "none")
show_register()

## 7. Build the graph from your registry (given)

Re-run this cell every time you change the registry. The summary shows how many links each pattern created: a pattern with **0 shared identifiers** adds nothing, and one that links *many* records deserves suspicion.

In [ ]:
def extract_entities(text):
    """Applies every registered pattern. Returns [(pattern_name, identifier), ...]."""
    found = []
    for p in PATTERNS:
        for m in re.finditer(p["regex"], text):
            ent = (p["name"], m.group(1))
            if ent not in found:
                found.append(ent)
    return found

assert not validate_registry(), "Fix the registry problems above first"

run_write("MATCH (n) DETACH DELETE n")

# One :Event node per record
event_rows = [{"id": f"R{i}", "text": t, "category": c}
              for i, (t, c) in enumerate(zip(rec_texts, rec_labels))]
run_write("""
    UNWIND $rows AS row
    CREATE (e:Event {id: row.id, text: row.text, category: row.category})
""", rows=event_rows)

# One entity node per distinct identifier; records naming it are linked through it
by_label = {}
for i, text in enumerate(rec_texts):
    for name, ident in extract_entities(text):
        by_label.setdefault(name, []).append({"eid": f"R{i}", "xid": ident})

for name, rows in by_label.items():       # name was validated: letters/digits/underscore only
    run_write(f"""
        UNWIND $rows AS r
        MATCH (e:Event {{id: r.eid}})
        MERGE (x:{name} {{id: r.xid}})
        MERGE (e)-[:MENTIONS]->(x)
    """, rows=rows)

print("Graph built from your registry:")
for name, rows in by_label.items():
    ids = {}
    for r in rows:
        ids.setdefault(r["xid"], 0)
        ids[r["xid"]] += 1
    shared = sum(1 for c in ids.values() if c > 1)
    print(f"  {name:<10} {len(rows):>3} mentions | {len(ids):>3} distinct identifiers | {shared} shared by 2+ records (= links)")

## 8. Grounded answer (given)

Retrieves, asks Neo4j (through `graph_check`) whether each pair of retrieved records is connected (and through which entity), then hands the LLM those facts with record IDs. `temperature=0`, so repeated runs give the same answer.

In [ ]:
def graph_check(id_a, id_b):
    """Asks Neo4j whether a path connects two records, and through which entities."""
    rows = run_read("""
        MATCH (a:Event {id: $a}), (b:Event {id: $b})
        OPTIONAL MATCH p = shortestPath((a)-[*..6]-(b))
        RETURN p IS NULL AS disconnected,
               [n IN nodes(p) WHERE NOT n:Event | labels(n)[0] + ' ' + n.id] AS via
    """, a=id_a, b=id_b)
    return rows[0] if rows else None


GROUNDED_PROMPT = """You answer questions about IT service and enterprise risk management (incidents, problems, risks, changes, audits) records. Use ONLY the records and graph facts below.

Rules:
1. The question's wording is a claim to check, NOT evidence. Do not accept a link just because the question assumes one.
2. User counts, rack numbers, severity levels, close periods, dates and similar shared values are NOT links.
3. The graph facts are ground truth about links. If the two records the question is about are NOT CONNECTED,
   say plainly that no recorded link exists between them. If they are CONNECTED, say so and name what they share.
4. If the records do not answer the question, say so. Cite record IDs like [R12].

Records:
{context}

Graph facts (checked in the database):
{graph_facts}

Question: {question}

Answer:"""


def grounded_answer(question, k=3):
    results = retrieve(question, k=k)
    ids = [f"R{idx}" for idx, _, _ in results]

    fact_lines, verified_connect, verified_disconnect = [], False, False
    for i in range(len(ids)):
        for j in range(i + 1, len(ids)):
            chk = graph_check(ids[i], ids[j])
            if chk is None:
                continue
            if chk["disconnected"]:
                verified_disconnect = True
                fact_lines.append(f"- [{ids[i]}] and [{ids[j]}]: NOT CONNECTED -- no shared named entity links them.")
            else:
                verified_connect = True
                fact_lines.append(f"- [{ids[i]}] and [{ids[j]}]: CONNECTED -- via {', '.join(chk['via'] or [])}.")

    context = "\n".join(f"- [R{idx}] {text}" for idx, text, _ in results)
    prompt = GROUNDED_PROMPT.format(context=context,
                                    graph_facts="\n".join(fact_lines) or "(no pairs to check)",
                                    question=question)
    return {"question": question, "retrieved": results, "graph_facts": fact_lines,
            "verified_connect": verified_connect, "verified_disconnect": verified_disconnect,
            "answer": call_llm(prompt)}

## 9. The public question pack

8 adversarial questions (the right answer is *no recorded connection*) and 3 positive controls (the right answer is *yes*). The judges will run a **different, hidden** pack of the same shapes, so fix your **registry**, not individual questions.

In [ ]:
PUBLIC_ADVERSARIAL = [{'id': 'P1',
  'question': 'Was the faulty switch replaced to resolve the network disruption that opened the '
              'major incident bridge?',
  'trap_records': ['A faulty switch was replaced and connectivity was restored',
                   'The major incident bridge was opened for a network disruption'],
  'correct_answer_must_state': 'No stated connection: neither record names an incident number or '
                               'shared entity.',
  'expect_connected': False,
  'kind': 'Vocabulary overlap (network)'},
 {'id': 'P2',
  'question': 'Did the review that found monitoring thresholds too high explain the recurring '
              'timeouts?',
  'trap_records': ['Recurring timeouts were grouped for a deeper investigation',
                   'The review found that monitoring thresholds had been set too high'],
  'correct_answer_must_state': 'No stated connection: the records name no shared problem or '
                               'incident.',
  'expect_connected': False,
  'kind': 'Vocabulary overlap (investigation)'},
 {'id': 'P3',
  'question': 'Was funding for the secondary network link approved because the risk was re-rated '
              'to high?',
  'trap_records': ['A risk was re-rated from medium to high',
                   'Funding was approved for a secondary network link'],
  'correct_answer_must_state': 'No stated connection: the re-rating names no risk ID, and the '
                               'funding record names none either.',
  'expect_connected': False,
  'kind': 'Vocabulary overlap (risk)'},
 {'id': 'P4',
  'question': 'Did tuning the duplicate alert rules reduce the phishing reports?',
  'trap_records': ['A phishing email was reported by several employees',
                   'Alert noise was reduced by tuning duplicate rules'],
  'correct_answer_must_state': 'No stated connection: monitoring noise and phishing reports are '
                               'separate records.',
  'expect_connected': False,
  'kind': 'Vocabulary overlap (alerts)'},
 {'id': 'P5',
  'question': 'Did the scheduled firmware update on Rack 14 trigger the cooling alarm?',
  'trap_records': ['Rack 14 was powered down for a scheduled firmware update',
                   'A cooling alarm was raised for Rack 14 in the server room'],
  'correct_answer_must_state': 'No stated connection: both mention Rack 14, but a rack is a place '
                               'reused by many events, not an event identifier.',
  'expect_connected': False,
  'kind': 'Kind 2 trap: rack'},
 {'id': 'P6',
  'question': 'Was the vendor audit finding rated Severity 2 because of the slow report generation '
              'ticket?',
  'trap_records': ['A vendor audit finding was rated Severity 2 by the review team',
                   'Severity 2 was assigned to the slow report generation ticket'],
  'correct_answer_must_state': 'No stated connection: a severity level is a rating shared by many '
                               'records.',
  'expect_connected': False,
  'kind': 'Kind 2 trap: severity'},
 {'id': 'P7',
  'question': 'Were the 300 users affected by the HR portal outage the same 300 users targeted by '
              'phishing?',
  'trap_records': ['An outage affected 300 users of the HR portal',
                   'A phishing campaign targeted 300 users in the sales team'],
  'correct_answer_must_state': 'No stated connection: the counts match by coincidence; the records '
                               'name different events.',
  'expect_connected': False,
  'kind': 'Kind 2 trap: user count'},
 {'id': 'P8',
  'question': 'Was the vendor questionnaire sent because the offsite backups were verified?',
  'trap_records': ['A vendor security questionnaire was sent for annual review',
                   'Offsite backups were verified for readability'],
  'correct_answer_must_state': 'No stated connection between a vendor review and a backup check; '
                               'any causal story is invented.',
  'expect_connected': False,
  'kind': 'Cross-category causal'}]

PUBLIC_POSITIVE = [{'id': 'PP1',
  'question': 'Was Incident INC4025 logged when card transactions stopped, and was it resolved by '
              'rolling back the gateway configuration?',
  'shared_identifier': 'Incident INC4025',
  'shared_records': ['Incident INC4025 was logged when the payment portal stopped accepting card '
                     'transactions',
                     'Incident INC4025 was resolved after the gateway configuration was rolled '
                     'back'],
  'correct_answer_must_state': 'Yes -- both records name Incident INC4025.',
  'expect_connected': True},
 {'id': 'PP2',
  'question': 'Was Incident INC4041 raised for failed logins and also used as evidence in the '
              'access review?',
  'shared_identifier': 'Incident INC4041',
  'shared_records': ['Incident INC4041 was raised after repeated failed logins from an unfamiliar '
                     'location',
                     'Incident INC4041 was included in the evidence for the access review'],
  'correct_answer_must_state': 'Yes -- both records name Incident INC4041.',
  'expect_connected': True},
 {'id': 'PP3',
  'question': 'Was Problem PRB114 opened for the booking timeouts, and was it traced to an expired '
              'service account?',
  'shared_identifier': 'Problem PRB114',
  'shared_records': ['Problem PRB114 was opened for the recurring timeouts on the booking service',
                     'Problem PRB114 was traced to an expired service account used by the booking '
                     'service'],
  'correct_answer_must_state': 'Yes -- both records name Problem PRB114.',
  'expect_connected': True}]

print(f"Public pack: {len(PUBLIC_ADVERSARIAL)} adversarial + {len(PUBLIC_POSITIVE)} positive-control questions")

In [ ]:
def structural_check(test):
    """No LLM needed: does the graph connect the two records this question is about?
    Returns (passed, connected, via)."""
    recs = test.get("trap_records") or test.get("shared_records")
    ia, ib = [f"R{rec_texts.index(t)}" for t in recs]
    chk = graph_check(ia, ib)
    connected = not chk["disconnected"]
    return connected == test["expect_connected"], connected, chk["via"]


def run_pack(pack, show_answers=True):
    """Runs baseline and grounded side by side on every question in a pack."""
    rows = []
    for t in pack:
        ok, connected, via = structural_check(t)
        print(f"[{t['id']}] {t['question']}")
        if show_answers:
            print("   Baseline :", baseline_rag_answer(t["question"])["answer"][:350].replace("\n", " "))
            print("   Grounded :", grounded_answer(t["question"])["answer"][:350].replace("\n", " "))
        print(f"   Graph    : {'CONNECTED' if connected else 'NOT connected'}"
              f"{' via ' + ', '.join(via) if via else ''} | expected "
              f"{'connected' if t['expect_connected'] else 'NOT connected'} -> {'PASS' if ok else 'FAIL'}")
        print("   Must say :", t["correct_answer_must_state"][:200])
        print()
        rows.append({"id": t["id"], "graph_pass": ok, "connected": connected})
    print(f"Graph verdict correct on {sum(r['graph_pass'] for r in rows)}/{len(rows)}")
    return rows

### Adversarial questions

In [ ]:
adv_rows = run_pack(PUBLIC_ADVERSARIAL)

### Positive controls: can your agent say *yes*?

In [ ]:
pos_rows = run_pack(PUBLIC_POSITIVE)

## 10. YOUR TASK: explain the outcomes

Fill in one line of `OUTCOMES` for each of the 11 public questions (P1-P8 and PP1-PP3). Each line has three fields. Read the **Baseline**, **Grounded**, **Graph** and **Must say** lines that the pack printed for that question, then decide:

| Field | The question it answers | How to decide |
| --- | --- | --- |
| `baseline_fabricated` | Did the plain model (no graph) **invent** something the records do not support? | Compare the **Baseline** answer with the **Must say** line. `True` only if the baseline claimed a link, a cause or "same person" that the records do not show. If it said "no", said "cannot tell", or said "yes" to a link that really exists, it did **not** fabricate: write `False`. |
| `grounded_correct` | Was **your** grounded answer right? | The **Graph** line ends in `PASS` -> `True`. `FAIL` -> `False`. |
| `explanation` | Why did it come out that way? | One sentence in terms of **your patterns**: which pattern matched (or did not), and which word was the shared ID, or a place, level, amount or period that is *not* an ID. |

**Most baselines are right on most questions.** `False` for `baseline_fabricated` is a normal, honest result; a long list of `True` that you did not actually see in the answers will lose marks. We score accurate, specific explanations, not the number of failures you found.

**How to write the explanation**

- Linked question (PP): *"Both records name <the ID>, and our <Name> pattern extracts it, so they are connected."*
- Not linked (P), shared word is a place, level, amount or period: *"Both mention <the word>, but that is a <place / amount / period>, a Kind 2 field reused by many events, not an event ID. We do not link on it."*
- Not linked (P), nothing shared: *"The two records name no shared ID, so no pattern links them."*
- Do not write "it might have..." or "it is not specified". That is guessing, and guessing is what the baseline does.

Example of the shape (not one of your questions):

```python
{"id": "P0", "baseline_fabricated": False, "grounded_correct": True,
 "explanation": "Both mention Shelf 3, but a shelf is a place, not an event ID, so no link."},
```

Then name one failure you still have in `REMAINING_FAILURE`. Be specific: a short honest note beats a vague claim.

In [ ]:
OUTCOMES = [
    # One dictionary per question, in this shape (see the table above):
    # {"id": "P1", "baseline_fabricated": ..., "grounded_correct": ..., "explanation": "..."},
    # Fill True or False for the first two fields; do not leave None.
    # Cover P1-P8 and PP1-PP3.
]

REMAINING_FAILURE = """
TODO: one case, or kind of case, where your system still gets it wrong (or could), and why.
"""


## 11. Submission summary

Run this last. This is what the judges read first.

In [ ]:
import json

def submission_summary():
    print("=== PATTERN REGISTER ===")
    show_register()
    # one machine-readable line the judges' tool reads: keep it
    print("REGISTRY_JSON:", json.dumps([{"name": p["name"], "regex": p["regex"]} for p in PATTERNS]))
    print("DATASET_ID:", DATASET_ID)
    print("\n=== OUTCOMES (public pack) ===")
    print(f"{'ID':<5} {'Baseline fabricated?':<22} {'Grounded correct?':<19} Explanation")
    for o in OUTCOMES:
        print(f"{o['id']:<5} {str(o['baseline_fabricated']):<22} {str(o['grounded_correct']):<19} {o['explanation']}")
    print("\n=== ONE FAILURE WE STILL HAVE ===")
    print(REMAINING_FAILURE)

submission_summary()

## 12. Before you submit

- Restart the runtime and **Run all**: it must work from a fresh state.
- `validate_registry()` reports no problems.
- `OUTCOMES` has an entry for every public question; `REMAINING_FAILURE` is filled in.
- Submit by the deadline announced on Day 1.